# INSTALL

In [ ]:
!pip install typhoon-ocr pdf2image Pillow

In [ ]:
!apt-get install -y poppler-utils

In [ ]:
!pip install -U google-generativeai

In [ ]:
!pip install python-dotenv

In [ ]:
# !pip uninstall -y transformers tokenizers huggingface_hub accelerate
!pip install transformers accelerate tiktoken einops

In [24]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

^C


In [15]:
# For CUDA 12.1 (check your driver version first)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp311-cp311-win_amd64.whl (2449.4 MB)
  Attempting uninstall: torch
    Found existing installation: torch 2.11.0
    Uninstalling torch-2.11.0:
      Successfully uninstalled torch-2.11.0


In [ ]:
import torch
print(torch.__version__)          
print(torch.cuda.is_available())  
print(torch.cuda.get_device_name(0))

2.11.0+cpu
False


AssertionError: Torch not compiled with CUDA enabled

# IMPORT

In [18]:
import os
import json
import re
import requests
# import dashscope
import shutil
from pdf2image import convert_from_path
from typhoon_ocr import ocr_document
from google import genai
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer

# Pipeline

In [12]:
class Extractor:
    # รายชื่อพรรคการเมืองที่ถูกต้องสำหรับใช้ตรวจสอบ (Validation)
    VALID_PARTIES = """
    ประชาธิปัตย์, ประชากรไทย, ความหวังใหม่, เครือข่ายชาวนาแห่งประเทศไทย, เพื่อไทย, 
    ชาติพัฒนา, ชาติไทยพัฒนา, อนาคตไทย, ภูมิใจไทย, สังคมประชาธิปไตยไทย, รักชาติ, 
    ประชาธิปไตยใหม่, พลังบูรพา, ครูไทยเพื่อประชาชน, พลังท้องถิ่นไท, ประชาชน, 
    ไทยก้าวใหม่, เสรีรวมไทย, รักษ์ธรรม, พลังประชาธิปไตย, พลังสุราษฎร์, พลังไทยรักชาติ, 
    เพื่อชีวิตใหม่, ทางเลือกใหม่, เศรษฐกิจ, สร้างอนาคตไทย, พลังธรรมใหม่, ไทยธรรม, 
    รวมพลัง, ไทยพร้อม, ปวงชนไทย, เพื่อชาติไทย, พร้อมพัฒนา, ประชาชาติ, แผ่นดินธรรม, 
    คลองไทย, พลังประชารัฐ, เศรษฐกิจใหม่, พลังสังคม, เป็นธรรม, พลังเพื่อไทย, ประชาไทย, 
    กรีน, วิชชั่นใหม่, พลวัต, กล้าธรรม, ไทยรวมไทย, กล้า, ฟิวชัน, พลังสังคมใหม่, 
    ไทยสร้างไทย, รวมไทยสร้างชาติ, มิติใหม่, ไทยสมาร์ท, ไทยภักดี, ไทยพิทักษ์ธรรม, 
    ไทยชนะ, ไทรวมพลัง, ราษฎร์วิถี, โอกาสใหม่, ท้องที่ไทย, ใหม่, แรงงานสร้างชาติ, 
    ไทยก้าวหน้า, ตะวันใหม่, พร้อม, รวมใจไทย, สัมมาธิปไตย, รักภูเก็ต, ประชาอาสาชาติ, 
    ไทยทรัพย์ทวี, รวมพลังประชาชน, อนาคตไกล, ยางพาราไทย, เพื่อบ้านเมือง
    """

    def __init__(self, typhoon_key: str, gemini_key: str = None, qwen_model_name: str = "Qwen/Qwen2.5-7B-Instruct"):
        """Initializes the OCR engine and the LLMs (Qwen and Gemini)."""
        # Setup Typhoon OCR
        os.environ["TYPHOON_OCR_API_KEY"] = typhoon_key
        
        # Setup Gemini (ใช้ Syntax ใหม่ของ google-genai)
        if gemini_key:
            self.gemini_client = genai.Client(api_key=gemini_key)
        else:
            self.gemini_client = None
        
        # Setup Qwen Local
        print(f"Loading Qwen model: {qwen_model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(qwen_model_name, trust_remote_code=True)
        
        self.qwen_model = AutoModelForCausalLM.from_pretrained(
            qwen_model_name,
            torch_dtype="auto",
            device_map="auto",
            trust_remote_code=True
        )

    def extract_pdf(self, pdf_path: str, use_model: str = "qwen") -> str:
        """Process PDF: Multi-page OCR -> Merged Text -> LLM Parsing."""
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"The file {pdf_path} was not found.")
        
        print(f"Starting Extraction: {pdf_path}")
        
        temp_dir = "temp_pages"
        if not os.path.exists(temp_dir):
            os.makedirs(temp_dir)
            
        full_markdown_text = ""
        
        try:
            # แปลง PDF เป็นภาพทีละหน้าเพื่อส่งให้ OCR
            pages = convert_from_path(pdf_path)
            
            for i, page in enumerate(pages):
                page_path = os.path.join(temp_dir, f"page_{i+1}.jpg")
                page.save(page_path, "JPEG")
                
                print(f"--- OCRing Page {i+1}/{len(pages)} ---")
                page_markdown = ocr_document(page_path)
                full_markdown_text += f"\n--- PAGE {i+1} ---\n" + page_markdown
            
            # เลือกว่าจะใช้ Model ตัวไหนในการ Parse ข้อมูล
            if use_model.lower() == "gemini" and self.gemini_client:
                print("Parsing with Gemini...")
                return self._parse_markdown_byGemini(full_markdown_text)
            else:
                print("Parsing with Qwen (Local)...")
                return self._parse_markdown_byQwen(full_markdown_text)
            
        finally:
            # ลบไฟล์ภาพชั่วคราว
            if os.path.exists(temp_dir):
                shutil.rmtree(temp_dir)

    def _get_shared_prompt(self, full_text: str) -> str:
        """Shared prompt logic for both models to ensure consistency."""
        return f"""
        Extract the election results from the following Thai OCR text into a structured JSON format.
        
        Requirements:
        1. Fix any OCR typos in province, district, or party names.
        2. Convert all numbers (including Thai digits) to standard integers.
        3. Include a 'metadata' section (province, district, unit), a 'summary' section (ballot counts), 
           and a 'results' list (party number, name, and votes).
        4. **PARTY NAME VALIDATION**: Compare the party name from OCR with the following valid list. 
           If the OCR name is misspelled, change it to the correct name from this list:
           {self.VALID_PARTIES}
        5. Check the score: look for numbers and Thai text in parentheses (). 
           If the main digit is unreadable, convert the Thai text description into an integer.

        OCR Text:
        {full_text}
        """

    def _parse_markdown_byGemini(self, full_text: str) -> str:
        prompt = self._get_shared_prompt(full_text)
        
        # ใช้ Syntax ใหม่ในการสั่ง Generate เนื้อหาและบังคับ JSON
        response = self.gemini_client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json"
            )
        )
        return response.text

    def _parse_markdown_byQwen(self, full_text: str) -> str:
        system_prompt = "You are Qwen, a helpful assistant created by Alibaba Cloud. You are an expert in Thai language and data extraction."
        user_prompt = self._get_shared_prompt(full_text)

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.qwen_model.device)

        # Generate using local model
        generated_ids = self.qwen_model.generate(
            **model_inputs,
            max_new_tokens=2048,
            temperature=0.1
        )

        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]

        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return self._clean_json_string(response)

    def _clean_json_string(self, content: str) -> str:
        """Removes Markdown code blocks from the string."""
        content = content.strip()
        if content.startswith("```json"):
            content = content[7:]
        elif content.startswith("```"):
            content = content[3:]
        if content.endswith("```"):
            content = content[:-3]
        return content.strip()

In [19]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: False
GPU count: 0


In [ ]:
if __name__ == "__main__":

    load_dotenv()
    
    TYPHOON_KEY = os.getenv("TYPHOON_KEY")
    GEMINI_KEY = os.getenv("GEMINI_KEY")
    os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
    extractor = Extractor(TYPHOON_KEY, GEMINI_KEY)
    
    try:
        json_output = extractor.extract_pdf("doc2.pdf")
        print("--- Extraction Result ---")
        print(json_output)
        
        # Save locally
        with open("output_gemini.json", "w", encoding="utf-8") as f:
            f.write(json_output)
            
    except Exception as e:
        print(f"An error occurred: {e}")

Loading Qwen model: Qwen/Qwen2.5-7B-Instruct...


Loading weights: 100%|██████████| 339/339 [00:00<00:00, 16140.36it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


Starting Extraction: doc2.pdf
--- OCRing Page 1/4 ---


KeyboardInterrupt: 

Looking in indexes: https://download.pytorch.org/whl/cu121
     ---------------------------------------- 0.0/2.4 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.4 GB 14.3 MB/s eta 0:02:52
     ---------------------------------------- 0.0/2.4 GB 13.8 MB/s eta 0:02:58
     ---------------------------------------- 0.0/2.4 GB 11.0 MB/s eta 0:03:42
     ---------------------------------------- 0.0/2.4 GB 9.8 MB/s eta 0:04:09
     ---------------------------------------- 0.0/2.4 GB 9.2 MB/s eta 0:04:24
     ---------------------------------------- 0.0/2.4 GB 9.1 MB/s eta 0:04:29
     ---------------------------------------- 0.0/2.4 GB 8.8 MB/s eta 0:04:38
     ---------------------------------------- 0.0/2.4 GB 8.6 MB/s eta 0:04:42
     ---------------------------------------- 0.0/2.4 GB 8.5 MB/s eta 0:04:46
     ---------------------------------------- 0.0/2.4 GB 8.4 MB/s eta 0:04:51
     ---------------------------------------- 0.0/2.4 GB 8.3 MB/s eta 0:04:53
     ----